# Vannettet: trykk, transienter og spenningskorrosjon

## Nettverksmatriser, vektor-ODE-er og en forenklet SCC-modell

### Pilotprosjekt for Matematikk 1, VVS

Et lite vannforsyningsnett leverer vann fra et reservoar til tre trykksoner. Nettet inneholder rette rør, et rørbend og en sveiset overgang. Pumpestans og endret forbruk gir trykkvariasjoner. I et kjemisk miljø som gjør materialet mottakelig for spenningskorrosjon, kan trykkindusert strekkspenning, residualspenning og lokal spenningskonsentrasjon bidra til sprekkvekst.

Prosjektet har fem deler:

1. **Stasjonært vannett:** forbindelsesmatrise, konduktanser og knutepunkttrykk
2. **Trykktank:** en skalar ODE med likevekt og tidskonstant
3. **Dynamisk trykknett:** et koblet vektor-ODE-system
4. **Trykkmoder:** diagonalisering og fysisk tolkning
5. **SCC-modell:** trykk, lokal spenning, spenningsintensitet og sprekkvekst

### Læringsmål

Etter prosjektet skal du kunne

- bygge en nettverksmatrise som $B^TGB$,
- forklare nullrommet til et vannett uten fast referansetrykk,
- sette inn et kjent reservoartrykk og løse $Kh=b$,
- beregne rørstrømmer og kontrollere massebalanse,
- modellere lekkasje som en ekstra konduktans,
- løse en skalar trykk-ODE analytisk og med Euler,
- skrive et trykknett som $C\dot h=s(t)-Kh$,
- diagonaliserer en symmetrisk nettverksmatrise,
- tolke trykkmoder og tidskonstanter,
- koble trykkhøyde til trykk og rørveggspenning,
- bruke en terskelmodell for SCC-sprekkvekst,
- diskutere forskjellen mellom hydraulisk kritikalitet og materialkritikalitet.

### Viktig avgrensning

Den hydrauliske rørmodellen er lineær og kan tolkes som en lokal linearisering rundt et driftspunkt. Trykktransientene er samlet i noen få trykknoder og er ikke en vannhammermodell med bølgeforplantning. SCC-loven er en pedagogisk terskelmodell. Den skal ikke brukes til dimensjonering, levetidsfastsettelse eller sikkerhetsvurdering av virkelige rør.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Referansenettet

Vi bruker fire trykknoder:

- node 0: reservoar og pumpe, kjent trykkhøyde $h_0$,
- node 1: sone A,
- node 2: sone B,
- node 3: sone C.

Rørene er:

```text
          rør 1             rør 2
  node 0 -------- node 1 -------- node 2
    |                |                |
    | rør 3          |                | rør 5
    |                |                |
  node 3 ----------- + ---------------+
          rør 4
```

Vi bruker kantrekkefølgen

$$
(0\!\to\!1),\quad(1\!\to\!2),\quad(0\!\to\!3),\quad(3\!\to\!1),\quad(2\!\to\!3).
$$

Rørstrømmen modelleres lineært:

$$
\boxed{q_e=g_e(h_i-h_j).}
$$

$h$ måles i meter vannsøyle og $q$ i $83{m^3/s}$.

In [ ]:
node_navn = ["reservoar", "sone A", "sone B", "sone C"]
rør_navn = ["rør 1", "rør 2", "rør 3", "rør 4", "rør 5"]

# Hydrauliske konduktanser i m^2/s: q = g * delta h
g = np.array([3.2e-4, 2.1e-4, 2.7e-4, 1.8e-4, 1.5e-4])

h0_normal = 48.0  # m vannsøyle
forbruk = np.array([5.0e-3, 3.2e-3, 2.6e-3])  # sone A, B, C i m^3/s

# Del A: Stasjonært vannett

## A.1 Forbindelsesmatrisen

Hver rad i $B$ representerer ett orientert rør. For et rør fra node $i$ til node $j$ setter vi

- $+1$ i kolonne $i$,
- $-1$ i kolonne $j$.

Da er vektoren av trykkhøydeforskjeller

$$
\boxed{\Delta h=Bh.}
$$

Rørstrømmene er

$$
\boxed{q=GBh,\qquad G=\operatorname{diag}(g_1,\ldots,g_5).}
$$

## Oppgave A1: Bygg $B$, $G$ og nettverksmatrisen

Vis at

$$
\boxed{K=B^TGB}
$$

er symmetrisk. Kontroller også at

$$K\mathbf1=0.$$

Forklar hvorfor alle trykkhøyder kan økes med samme konstant uten å endre rørstrømmene.

In [ ]:
B = np.array([
    [ 1.0, -1.0,  0.0,  0.0],
    [ 0.0,  1.0, -1.0,  0.0],
    [ 1.0,  0.0,  0.0, -1.0],
    [ 0.0, -1.0,  0.0,  1.0],
    [ 0.0,  0.0,  1.0, -1.0]
])

G = ...
K_full = ...

print("B =
", B)
print("K_full =
", K_full)
print("Symmetrisk:", ...)
print("K_full @ 1 =", ...)
print("Egenverdier:", ...)

## A.2 Fest reservoartrykket

Skriv

$$h=(h_0,h_1,h_2,h_3)^T.$$

Reservoartrykket $h_0$ er kjent, mens de tre sonehøydene er ukjente. Kontinuitet i de ukjente nodene gir

$$
\boxed{K_{uu}h_u=b_u-K_{u0}h_0.}
$$

Fortegnskonvensjonen er at positivt forbruk trekker vann ut av noden.

## Oppgave A2: Løs nodehøydene

Bruk de tre siste radene og kolonnene i $K_{full}$. Høyresiden for de ukjente nodene er $-d$, der $d$ er forbruksvektoren.

In [ ]:
K_uu = K_full[1:, 1:]
K_u0 = K_full[1:, 0]

b_u = -forbruk
h_u = ...
h = np.concatenate([[h0_normal], h_u])

print("Trykkhøyder:", h)
print("Residual i ukjente noder:", ...)

## Oppgave A3: Rørstrømmer og massebalanse

Beregn

$$q=GBh.$$

Et negativt element betyr at den virkelige strømretningen er motsatt den valgte rørorienteringen.

Kontroller knutepunktbalansen med

$$B^Tq.$$

In [ ]:
q_rør = ...
nodebalanse = ...

for navn, qi in zip(rør_navn, q_rør):
    print(f"{navn:7s}: {1000*qi:8.3f} L/s")
print("Nodebalanse:", nodebalanse)
print("Forventet i forbruksnoder:", -forbruk)

## A.3 Lekkasjer

En linearisert lekkasje ved node $i$ kan modelleres som

$$q_{lekk,i}=\ell_i h_i.$$

Lekkasjene legges til som diagonalmatrisen

$$L=\operatorname{diag}(\ell_1,\ell_2,\ell_3).$$

Da blir det reduserte systemet

$$
\boxed{(K_{uu}+L)h_u=b_u-K_{u0}h_0.}
$$

## Oppgave A4: Lekkasjested og robusthet

Sammenlign:

1. ingen lekkasje,
2. lekkasje i sone B,
3. lekkasje i sone C,
4. rør 5 stengt,
5. rør 2 stengt.

For hvert tilfelle, rapporter laveste trykkhøyde og endringen i rørstrømmene.

In [ ]:
lekkasje = np.array([0.0, 4.0e-5, 0.0])
L_lekk = ...
h_u_lekk = ...
print("Trykkhøyder med lekkasje:", h_u_lekk)

# Del B: En trykktank som skalar ODE

En trykktank er koblet til et reservoar gjennom et rør. Trykkhøyden i tanknoden er $h(t)$. En lineær last trekker volumstrøm $d h$.

Balansen er

$$
\boxed{C_h\dot h=g(h_0-h)-dh.}
$$

Her er $C_h$ en hydraulisk kapasitet med enhet $83{m^2}$ i denne aggregerte modellen.

In [ ]:
C_h = 0.025
g_tank = 3.0e-4
d_last = 7.0e-5
h_res = 48.0

## Oppgave B1: Likevekt og tidskonstant

Skriv ligningen som

$$\dot h+ah=b.$$

Finn

$$h^*=\frac{b}{a},\qquad \tau=\frac1a.$$

Løs også ligningen analytisk for startverdien $h(0)=30$ m.

In [ ]:
a_tank = ...
b_tank = ...
h_tank_likevekt = ...
tau_tank = ...

print("Likevekt:", h_tank_likevekt)
print("Tidskonstant:", tau_tank, "s")


def h_analytisk(t, h_start=30.0):
    return ...

## Oppgave B2: Euler mot analytisk løsning

Sammenlign flere tidssteg. Undersøk hva som skjer dersom forbruket dobles brått.

In [ ]:
def euler_skalar(f, y0, sluttid, dt):
    n = int(round(sluttid/dt))
    t = np.linspace(0, n*dt, n+1)
    y = np.zeros(n+1)
    y[0] = y0
    for k in range(n):
        y[k+1] = ...
    return t, y


def tank_ode(t, h):
    return (g_tank*(h_res-h)-d_last*h)/C_h

# Kjør og plott mot h_analytisk.

# Del C: Dynamisk trykknett

## C.1 Tre hydrauliske kapasiteter

De tre trykksonene får kapasitetene

$$
C=\operatorname{diag}(C_1,C_2,C_3).
$$

Dynamikken er

$$
\boxed{C\dot h_u=s(t)-K_{uu}h_u.}
$$

Kildetermen er

$$s(t)=-d(t)-K_{u0}h_0(t).$$

Den stasjonære løsningen er den samme som i del A.

In [ ]:
C_node = np.diag([0.035, 0.025, 0.030])


def reservoarhøyde(t_s):
    # Pumpeutfall mellom 8 og 14 sekunder.
    if 8.0 <= t_s < 14.0:
        return 34.0
    return h0_normal


def forbruk_tid(t_s):
    d = forbruk.copy()
    # Midlertidig økt uttak i sone B.
    if 18.0 <= t_s < 24.0:
        d[1] *= 1.7
    return d


def trykk_ode(t_s, h_u):
    s = -forbruk_tid(t_s) - K_u0*reservoarhøyde(t_s)
    return ...

## Oppgave C1: Euler for vektorsystemet

Start i den stasjonære normaltilstanden. Simuler 30 sekunder og plott alle tre trykkhøydene.

In [ ]:
def euler_system(f, x0, sluttid, dt):
    n = int(round(sluttid/dt))
    t = np.linspace(0, n*dt, n+1)
    X = np.zeros((n+1, len(x0)))
    X[0] = x0
    for k in range(n):
        X[k+1] = ...
    return t, X


t_C, h_C = euler_system(trykk_ode, h_u, 30.0, 0.01)

for j in range(3):
    plt.plot(t_C, h_C[:, j], label=node_navn[j+1])
plt.xlabel("Tid s")
plt.ylabel("Trykkhøyde m")
plt.grid(); plt.legend(); plt.show()

## Oppgave C2: Trykktanker og lekkasje

Undersøk virkningen av

- større kapasitet i sone B,
- lik kapasitet i alle soner,
- lekkasje i sone B,
- stengt ringforbindelse,
- kortere og lengre pumpeutfall.

Sammenlign minimumstrykk og innstillingstid.

# Del D: Diagonalisering og trykkmoder

For en ren modalanalyse bruker vi tre like kapasiteter og det symmetriske nettet

$$
K_{sym}=
\begin{pmatrix}
g_0+g&-g&0\\
-g&g_0+2g&-g\\
0&-g&g_0+g
\end{pmatrix}.
$$

$g$ kobler nabosonene, mens $g_0$ kobler hver sone til et referansesystem eller representerer lineær last/lekkasje.

In [ ]:
C0 = 0.030
g0 = 1.0e-4
gm = 2.4e-4

K_sym = np.array([
    [g0+gm, -gm, 0.0],
    [-gm, g0+2*gm, -gm],
    [0.0, -gm, g0+gm]
])

## Oppgave D1: Finn trykkmodene

Diagonaliser

$$K_{sym}=P\Lambda P^T.$$

Tolk modene som

- felles trykkendring,
- forskjell mellom endesonene,
- midtsonen mot endesonene.

In [ ]:
egenverdier_K, P = ...
Lambda = ...

print("Egenverdier:", egenverdier_K)
print("Egenvektorer:
", P)
print("P^T P:
", ...)
print("P Lambda P^T:
", ...)

## D.2 Frakoblede ODE-er

Sett trykkavviket fra likevekt til

$$h-h^*=Pz.$$

Da får vi

$$
\boxed{C_0\dot z=-\Lambda z.}
$$

Tidskonstantene er

$$
\boxed{\tau_i=\frac{C_0}{\lambda_i}.}
$$

## Oppgave D2: Tidskonstanter og modal simulering

1. Beregn tidskonstantene.
2. Start med høyt trykk bare i sone A.
3. Simuler direkte og i modale koordinater.
4. Transformer tilbake og kontroller at løsningene stemmer.
5. Forklar hvilke moder som utjevnes raskest.

In [ ]:
tau = ...
print("Tidskonstanter i sekunder:", tau)

# Implementer direkte og modal simulering.

# Del E: Spenningskorrosjon ved tre komponenter

## E.1 Tre kritiske steder

Vi analyserer:

1. et rett rør,
2. et 90-graders bend,
3. en sveiset overgang.

Komponentene har samme nominelle diameter og veggtykkelse, men ulike lokale spennings- og residualspenningsfaktorer.

Trykkhøyden omformes til overtrykk:

$$
\boxed{p=\rho g h.}
$$

For et tynnvegget rett rør er omkretsspenningen

$$
\boxed{\sigma_\theta=\frac{pD}{2w}.}
$$

Den lokale effektive strekkspenningen modelleres som

$$
\boxed{\sigma_i=K_{\sigma,i}\frac{p_iD_i}{2w_i}+\sigma_{res,i}+\sigma_{bøy,i}.}
$$

$K_{\sigma}$ er her en pedagogisk lokal faktor, ikke en komplett spenningsanalyse av et bend eller en sveis.

In [ ]:
komponent_navn = ["rett rør", "bend", "sveis"]

rho_vann = 998.0
g_tyngde = 9.81
D_rør = np.array([0.100, 0.100, 0.100])  # m
veggtykkelse = np.array([0.0045, 0.0045, 0.0045])  # m

K_sigma = np.array([1.00, 1.45, 1.20])
sigma_res = np.array([20e6, 70e6, 110e6])  # Pa
sigma_bøy = np.array([0.0, 25e6, 10e6])

# Knytt komponentene til sonene A, B, C.
h_drift = h_u.copy()

## Oppgave E1: Beregn lokal spenning

Beregn trykk og lokal spenning ved normalt driftspunkt. Sammenlign rangeringen med rangeringen av nodehøydene.

In [ ]:
p_drift = ...
sigma_trykk = ...
sigma_lokal = ...

for navn, hi, pi, si in zip(komponent_navn, h_drift, p_drift, sigma_lokal):
    print(f"{navn:10s}: h={hi:6.2f} m, p={pi/1e5:6.2f} bar, sigma={si/1e6:7.2f} MPa")

## E.2 Spenningsintensitet

For en idealisert sprekk med dybde $a$ bruker vi

$$
\boxed{K_I=Y\sigma\sqrt{\pi a}.}
$$

$Y$ er en geometriparameter. Denne enkle formelen samler ikke alle effekter av virkelig sprekkform og rørgeometri.

In [ ]:
a0 = np.array([0.20e-3, 0.20e-3, 0.20e-3])  # m
Y_sprekk = np.array([1.05, 1.20, 1.15])

K_I0 = ...
print("K_I i MPa sqrt(m):", K_I0/1e6)

## E.3 Pedagogisk SCC-vekstlov

Vi bruker

$$
\boxed{
\dot a_i
=C_iE_i\left[\max(0,K_{I,i}-K_{ISCC,i})\right]^{n_i}.}
$$

I kode skalerer vi $K_I$ til MPa$\sqrt{\mathrm m}$ for å få håndterlige parametertall. Miljøfaktoren $E_i$ er null dersom SCC-miljøet ikke er aktivt.

Parameterne er valgt for undervisningsformål og representerer ikke et bestemt materiale eller vannsystem.

In [ ]:
K_ISCC = np.array([18.0, 18.0, 18.0])  # MPa sqrt(m)
C_scc = np.array([1.2e-10, 1.2e-10, 1.2e-10])  # m/s per skalert drivkraft
n_scc = 1.25
miljøfaktor = np.array([1.0, 1.0, 1.0])


def lokal_spenning(h, a=None):
    p = rho_vann*g_tyngde*np.asarray(h)
    return K_sigma*p*D_rør/(2*veggtykkelse) + sigma_res + sigma_bøy


def scc_ode(t_s, a, h_representativ=h_drift):
    sigma = lokal_spenning(h_representativ)
    K_I_MPa = Y_sprekk*sigma*np.sqrt(np.pi*a)/1e6
    drivkraft = np.maximum(0.0, K_I_MPa-K_ISCC)
    return C_scc*miljøfaktor*drivkraft**n_scc

## Oppgave E2: Sprekkvekst over tid

Simuler de tre sprekkene med Euler. Bruk en passende langsom tidsenhet og stopp dersom

$$a_i\geq0.8w_i.$$

Plott sprekkdybden som andel av veggtykkelsen. Hvilken komponent blir kritisk først?

In [ ]:
def euler_scc(f, a_start, sluttid, dt):
    n = int(round(sluttid/dt))
    t = np.linspace(0, n*dt, n+1)
    A = np.zeros((n+1, len(a_start)))
    A[0] = a_start

    for k in range(n):
        A[k+1] = ...
        if np.any(A[k+1] >= 0.8*veggtykkelse):
            return t[:k+2], A[:k+2]
    return t, A

# Velg sluttid og tidssteg etter å ha undersøkt startveksthastigheten.
print("Startveksthastighet m/s:", scc_ode(0.0, a0))

## E.4 Kobling til trykktransienten

For en kort trykktransient kan den øyeblikkelige spenningen beregnes fra $h_i(t)$. SCC virker normalt på en langt langsommere tidsskala enn trykktransienten.

En enkel fler-skala-tilnærming er:

1. simuler trykket over én representativ driftssyklus,
2. beregn middelverdien av SCC-veksthastigheten gjennom syklusen,
3. bruk denne som langsom årlig eller månedlig veksthastighet.

Dette må skilles fra korrosjonsutmattelse, som er knyttet til syklisk last i et korrosivt miljø.

## Oppgave E3: Normaltrykk, trykkreduksjon og bendutskifting

Sammenlign:

- normal drift,
- 15 prosent lavere reservoartrykk,
- redusert residualspenning,
- nytt bend med lavere $K_\sigma$,
- mindre startsprekk,
- miljøfaktor lik null.

Rapporter hydrauliske konsekvenser og beregnet SCC-utvikling. Diskuter hvorfor det hydraulisk viktigste røret ikke nødvendigvis er komponenten med høyest SCC-risiko.

# Valgfri utvidelse: aldring påvirker hydraulikken

Generell korrosjon, avleiringer og ruhetsøkning kan redusere en rørgrens hydrauliske kapasitet. En enkel pedagogisk modell er

$$
\boxed{\dot g_i=-\alpha_i r_i g_i.}
$$

Ved hvert langsomme tidssteg:

1. oppdater $g_i$,
2. bygg $G(t)$ og $K(t)=B^TG(t)B$,
3. løs det stasjonære vannettet,
4. beregn trykk og lokal spenning,
5. oppdater sprekkdybdene.

Dette gir et lineært hydraulisk system inne i hvert ODE-steg. Modellen bør ikke blandes sammen med SCC-loven: generell korrosjon, avleiring og SCC er forskjellige mekanismer.

# Modellkritikk

Diskuter minst seks punkter:

- Rørstrømmen er lineær i trykkhøydeforskjellen.
- Turbulent trykktap er vanligvis ikke lineært.
- Kapasitetsnodene er aggregerte og beskriver ikke vannhammerbølger.
- Lekkasjemodellen er lineær.
- Reservoar- og pumpehøyden er sterkt forenklet.
- Rørnettets geometri er redusert til noen få noder.
- Tynnvegget rørteori er en tilnærming.
- $K_\sigma$ samler komplisert lokal geometri i ett tall.
- Residualspenningene er oppgitte scenarioverdier.
- Virkelig SCC er material- og miljøspesifikk.
- En enkel $K_I$-formel beskriver ikke en virkelig sprekk i et rørbend fullstendig.
- SCC-parameterne er syntetiske undervisningsparametre.
- SCC og korrosjonsutmattelse må skilles.
- Sprekkveksten påvirker ikke lekkasje eller hydraulikk i hovedmodellen.
- Bruddkriteriet $a=0.8w$ er bare en stoppindikator, ikke et godkjent integritetskriterium.
- Virkelige rør krever standarder, materialdata, inspeksjon og fagkyndig integritetsanalyse.

## Mulige videreføringer

- Darcy–Weisbach eller Hazen–Williams med ikke-lineær løsning,
- pumpekurve og driftskurve,
- flere reservoarer og ventiler,
- trykkstyrt lekkasje,
- vannhammer i et senere emne,
- temperatur- og kjemiavhengig SCC-modell,
- korrosjonsutmattelse ved syklisk trykk,
- pålitelighet og usikre materialparametre,
- inspeksjonsdata og prioritering av vedlikehold.

# Oppsummering

Skriv en kort rapport der du forklarer

1. hvordan forbindelsesmatrisen beskrev vannettet,
2. hvorfor $B^TGB$ var singulær før referansetrykket ble festet,
3. hvordan nodehøyder og rørstrømmer ble beregnet,
4. hvordan lekkasje og rørstenging påvirket nettet,
5. hvordan trykktanken ga en skalar ODE,
6. hvordan trykknettet ble et vektor-ODE-system,
7. hva trykkmodene og tidskonstantene betyr fysisk,
8. hvordan trykkhøyde ble koblet til lokal rørspenning,
9. hvordan sprekkdybde påvirket spenningsintensiteten,
10. hvorfor SCC krever både et mottakelig materiale, et relevant miljø og strekkspenning.

## Faglig bakgrunn

Prosjektet bruker standard nettverksidéer, en tynnvegget trykkrørtilnærming og en idealisert spenningsintensitetsmodell. Rørbend og sveisede fittings kan ha lokale og residuale spenninger som gjør dem mer utsatte enn rette rør under bestemte material- og miljøforhold.

Studentene trenger ikke eksterne kilder for hovedløpet. Selvvalgte material- eller SCC-parametre i fordypningen skal dokumenteres.